###Импорт библиотек и загрузка данных

In [2]:
import pandas as pd
import numpy as np
import altair as alt

alt.renderers.enable('colab')

DATA_PATH = "person_2025_update.csv.bz2"

df = pd.read_csv(DATA_PATH, low_memory=False, compression='bz2')

num_cols = ['birthyear','deathyear','hpi','hpi_raw','age','non_en_page_views','prob_ratio','coefficient_of_variation']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

for date_col in ['birthdate','deathdate']:
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

if 'birthyear' in df.columns:
    df['birth_decade'] = (df['birthyear']//10)*10
if 'birthyear' in df.columns and 'deathyear' in df.columns:
    df['computed_age'] = df['deathyear'] - df['birthyear']
if 'gender' in df.columns:
    df['gender'] = df['gender'].fillna('unknown').astype(str)
if 'occupation' in df.columns:
    df['occupation'] = df['occupation'].fillna('UNKNOWN').astype(str)

print("Данные загружены и подготовлены.")

/tmp/ipython-input-2885785971.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
/tmp/ipython-input-2885785971.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col], errors='coerce')


Данные загружены и подготовлены.


---

###График 1 — Гистограммы годов рождения (Pre-1800 и Post-1800)

In [3]:
pre1800 = df[df['birthyear'] < 1800]['birthyear'].dropna()
hist_pre, bins_pre = np.histogram(pre1800, bins=range(int(pre1800.min())//100*100, 1801, 100))
data_pre = pd.DataFrame({'year': bins_pre[:-1], 'count': hist_pre})

chart1 = alt.Chart(data_pre).mark_bar(color='skyblue', stroke='black').encode(
    x=alt.X('year:O', title='Year of birth'),
    y=alt.Y('count:Q', title='Count'),
    tooltip=['year', 'count']
).properties(title='Before 1800 (100-year bins)', width=300)

post1800 = df[df['birthyear'] >= 1800]['birthyear'].dropna()
hist_post, bins_post = np.histogram(post1800, bins=range(1800, int(post1800.max())+10, 10))
data_post = pd.DataFrame({'year': bins_post[:-1], 'count': hist_post})

chart2 = alt.Chart(data_post).mark_bar(color='salmon', stroke='black').encode(
    x=alt.X('year:O', title='Year of birth'),
    y=alt.Y('count:Q', title='Count'),
    tooltip=['year', 'count']
).properties(title='From 1800 onwards (10-year bins)', width=300)

chart1 | chart2

alt.HConcatChart(...)

---

###График 2 — Топ-10 профессий

In [5]:
top_occ = df['occupation'].value_counts().nlargest(10).reset_index()
top_occ.columns = ['occupation', 'count']

alt.Chart(top_occ).mark_bar().encode(
    x=alt.X('count:Q', title='Count'),
    y=alt.Y('occupation:N', sort='-x', title='Occupation'),
    tooltip=['occupation', 'count']
).properties(title='Top 10 occupations in dataset')

alt.Chart(...)

---

###График 3 — Распределение по полу (Pie Chart)

In [6]:
gender_counts = df['gender'].value_counts().reset_index()
gender_counts.columns = ['gender', 'count']

alt.Chart(gender_counts).mark_arc(outerRadius=100).encode(
    theta=alt.Theta("count", stack=True),
    color=alt.Color("gender"),
    tooltip=["gender", "count", alt.Tooltip("count", format=".1%")] # Добавил проценты в подсказку
).properties(title='Gender distribution')

alt.Chart(...)

---

###График 4 — Топ-20 стран рождения

In [7]:
country_counts = df['bplace_country'].value_counts().head(20).reset_index()
country_counts.columns = ['country', 'count']

alt.Chart(country_counts).mark_line(point=True, color='green').encode(
    x=alt.X('country', sort=None, title='Birth country'),
    y=alt.Y('count', title='Number of persons'),
    tooltip=['country', 'count']
).properties(title='Number of persons by birth country (Top 20)', width=600)

alt.Chart(...)

---

###График 5 — Strip Plot (Gender vs Non-en views)


In [9]:
sub = df[df['non_en_page_views'].notna() & df['gender'].notna()].sample(n=min(1000, len(df)), random_state=42)

alt.Chart(sub).mark_circle(size=8, opacity=0.6).encode(
    x=alt.X('gender', title='Gender'),
    y=alt.Y('non_en_page_views', scale=alt.Scale(type='log'), title='Non-English page views (log)'),
    color='gender',
    tooltip=['gender', 'non_en_page_views']
).properties(title='Distribution of non-English page views by gender', width=400)

alt.Chart(...)

---

###График 6 — Scatter Plot (HPI vs Log Views)


In [8]:
sub_hpi = df[(df['non_en_page_views'] > 0) & df['hpi'].notna()].sample(n=min(1000, len(df)), random_state=42)

alt.Chart(sub_hpi).mark_circle(size=20, opacity=0.5).encode(
    x=alt.X('non_en_page_views', scale=alt.Scale(type='log'), title='log10(non_en_page_views)'),
    y=alt.Y('hpi', title='HPI'),
    tooltip=['non_en_page_views', 'hpi']
).properties(title='HPI vs log(non_en_page_views)', width=400)

alt.Chart(...)

---

###График 7 — Рождения по месяцам

In [11]:
df['month'] = df['birthdate'].dt.month
month_counts = df['month'].value_counts().sort_index().reset_index()
month_counts.columns = ['month', 'count']

import calendar
month_map = {i: calendar.month_abbr[i] for i in range(1, 13)}
month_counts['month_name'] = month_counts['month'].map(month_map)

alt.Chart(month_counts).mark_bar().encode(
    x=alt.X('month_name', sort=list(month_map.values()), title='Month of Birth'),
    y=alt.Y('count', title='Number of Persons'),
    color=alt.value('steelblue'), # Чтобы было похоже на coolwarm, можно использовать scale, но steelblue проще
    tooltip=['month_name', 'count']
).properties(title='Number of Persons Born by Month', width=500)

alt.Chart(...)

---

###График 8 — KDE Plot для HPI

In [13]:
kde_data = df[df['hpi'].notna()].sample(n=min(2000, len(df)), random_state=42)

alt.Chart(kde_data).transform_density(
    'hpi',
    as_=['hpi', 'density']
).mark_area(opacity=0.5).encode(
    x=alt.X('hpi', title='HPI'),
    y='density:Q',
).properties(title='Kernel density estimate of HPI')

alt.Chart(...)

---

###  График 9 — Матрица корреляций

In [17]:
num_cols = ['hpi','hpi_raw','birthyear','deathyear','computed_age','non_en_page_views','prob_ratio','coefficient_of_variation']
exist_cols = [c for c in num_cols if c in df.columns]

corr_matrix = df[exist_cols].corr().reset_index().melt(id_vars='index')

base = alt.Chart(corr_matrix).encode(
    x=alt.X('index:N', title=None),
    y=alt.Y('variable:N', title=None)
).properties(width=400, height=400, title='Correlation matrix')

heatmap = base.mark_rect().encode(
    color=alt.Color('value:Q',
                    scale=alt.Scale(scheme='redblue', domain=[-1, 1]),
                    title="Correlation")
)

text = base.mark_text().encode(
    text=alt.Text('value:Q', format='.2f'),
    color=alt.condition(
        'abs(datum.value) > 0.5',
        alt.value('white'),
        alt.value('black')
    )
)

heatmap + text

alt.LayerChart(...)

---

###График 10 — Карта (Scatter Map)

In [18]:
sub_geo = df[df['bplace_lon'].notna() & df['bplace_lat'].notna()].sample(n=min(2000, len(df)), random_state=42)

alt.Chart(sub_geo).mark_circle(size=15).encode(
    longitude='bplace_lon',
    latitude='bplace_lat',
    color=alt.Color('hpi', scale=alt.Scale(scheme='viridis')),
    tooltip=['bplace_country', 'hpi']
).project('naturalEarth1').properties(
    title='Birthplace coordinates colored by HPI (Sampled)',
    width=700,
    height=400
)

alt.Chart(...)

### Опыт работы с Altair

В ходе выполнения работы я перенес визуализации из библиотек Matplotlib/Seaborn в Altair. Мои выводы:

* **Интерактивность:** Это главное преимущество. Графики стали «живыми»: теперь можно наводить курсор на бары или точки и видеть точные значения во всплывающих подсказках. Это намного удобнее для анализа, чем статические картинки.
* **Декларативный подход:** Код пишется очень логично — мы просто описываем, какие данные к каким визуальным свойствам (ось, цвет, размер) привязаны. Комбинировать графики между собой через | и + гораздо проще, чем возиться с subplots.
* **Работа с данными:** Altair заметно капризнее к объему данных. В отличие от Matplotlib, он встраивает данные прямо в файл блокнота. Из-за этого Google Colab может выдавать ошибки при сохранении. Чтобы всё работало стабильно, пришлось заранее агрегировать данные в Pandas или делать небольшие случайные выборки.